In [1]:
import pandas as pd
from math import sqrt
import numpy as np
import matplotlib.pyplot as plt

In [7]:
movies_df = pd.read_csv('movies.csv')
ratings_df = pd.read_csv('ratings.csv')

In [ ]:
movies_df['year'] = movies_df.title.str.extract('(\(\d\d\d\d\))', expand=False)
movies_df['year'] = movies_df.year.str.extract('(\d\d\d\d)', expand=False)
movies_df['title'] = movies_df.title.str.replace('(\(\d\d\d\d\))', '')
movies_df['title'] = movies_df['title'].apply(lambda x: x.strip())
movies_df

In [10]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [11]:
movies_df['genres'] = movies_df.genres.str.split('|')

In [12]:
movies_df

,movieId,title,genres
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]"
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]"
2,3,Grumpier Old Men (1995),"[Comedy, Romance]"
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]"
4,5,Father of the Bride Part II (1995),[Comedy]
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]"
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]"
9739,193585,Flint (2017),[Drama]
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]"


In [13]:
moviesWithGenres_df = movies_df.copy()
for index, row in movies_df.iterrows():
    for genre in row['genres']:
        moviesWithGenres_df.at[index, genre] = 1
moviesWithGenres_df = moviesWithGenres_df.fillna(0)

In [14]:
moviesWithGenres_df

,movieId,title,genres,Adventure,Animation,Children,Comedy,Fantasy,Romance,Drama,...,Horror,Mystery,Sci-Fi,War,Musical,Documentary,IMAX,Western,Film-Noir,(no genres listed)
0,1,Toy Story (1995),"[Adventure, Animation, Children, Comedy, Fantasy]",1.0,1.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2,Jumanji (1995),"[Adventure, Children, Fantasy]",1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,Grumpier Old Men (1995),"[Comedy, Romance]",0.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,Waiting to Exhale (1995),"[Comedy, Drama, Romance]",0.0,0.0,0.0,1.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,Father of the Bride Part II (1995),[Comedy],0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),"[Action, Animation, Comedy, Fantasy]",0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9738,193583,No Game No Life: Zero (2017),"[Animation, Comedy, Fantasy]",0.0,1.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9739,193585,Flint (2017),[Drama],0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9740,193587,Bungo Stray Dogs: Dead Apple (2018),"[Action, Animation]",0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [61]:
ratings_df

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [15]:
# حذف ستون timestamp که نیاز نداریم
ratings_df = ratings_df.drop('timestamp', axis=1)

In [16]:
ratings_df

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
2,1,6,4.0
3,1,47,5.0
4,1,50,5.0
...,...,...,...
100831,610,166534,4.0
100832,610,168248,5.0
100833,610,168250,5.0
100834,610,168252,5.0


In [17]:
userInput = [
    {'title':'Toy Story', 'rating':3.5},
    {'title':'Jumanji', 'rating':2},
    {'title':'Pulp Fiction', 'rating':5},
    {'title':'Akira', 'rating':4.5}
]
inputMovies = pd.DataFrame(userInput)

In [18]:
inputMovies

,title,rating
0,Toy Story,3.5
1,Jumanji,2.0
2,Pulp Fiction,5.0
3,Akira,4.5


In [20]:


# Sample movies_df for demonstration
movies_data = [
    {'movieId': 1, 'title': 'Toy Story', 'genres': 'Animation', 'year': 1995},
    {'movieId': 2, 'title': 'Jumanji', 'genres': 'Adventure', 'year': 1995},
    {'movieId': 3, 'title': 'Pulp Fiction', 'genres': 'Crime', 'year': 1994},
    {'movieId': 4, 'title': 'Akira', 'genres': 'Animation', 'year': 1988}
]
movies_df = pd.DataFrame(movies_data)

# User input
userInput = [
    {'title':'Toy Story', 'rating':3.5},
    {'title':'Jumanji', 'rating':2},
    {'title':'Pulp Fiction', 'rating':5},
    {'title':'Akira', 'rating':4.5}
]
inputMovies = pd.DataFrame(userInput)

# Merging
inputId = movies_df[movies_df['title'].isin(inputMovies['title'].tolist())]
inputMovies = pd.merge(inputId, inputMovies, on='title').drop('genres', axis=1).drop('year', axis=1)

# Display result
print(inputMovies)


   movieId         title  rating
0        1     Toy Story     3.5
1        2       Jumanji     2.0
2        3  Pulp Fiction     5.0
3        4         Akira     4.5


In [22]:
inputId

,movieId,title,genres,year
0,1,Toy Story,Animation,1995
1,2,Jumanji,Adventure,1995
2,3,Pulp Fiction,Crime,1994
3,4,Akira,Animation,1988


In [21]:
inputMovies

,movieId,title,rating
0,1,Toy Story,3.5
1,2,Jumanji,2.0
2,3,Pulp Fiction,5.0
3,4,Akira,4.5


In [23]:
userSubset = ratings_df[ratings_df['movieId'].isin(inputMovies['movieId'].tolist())]
userSubsetGroup = userSubset.groupby(['userId'])

In [24]:
userSubset

,userId,movieId,rating
0,1,1,4.0
1,1,3,4.0
516,5,1,4.0
560,6,2,4.0
561,6,3,5.0
...,...,...,...
98666,608,1,2.5
98667,608,2,2.0
98668,608,3,2.0
99497,609,1,3.0


In [25]:
pearsonCorrelationDict = {}
for name, group in userSubsetGroup:
    group = group.sort_values(by='movieId')
    inputMovies = inputMovies.sort_values(by='movieId')
    nRatings = len(group)
    temp_df = inputMovies[inputMovies['movieId'].isin(group['movieId'].tolist())]
    tempRatingList = temp_df['rating'].tolist()
    tempGroupList = group['rating'].tolist()
    
    Sxx = sum([i**2 for i in tempRatingList]) - pow(sum(tempRatingList),2)/float(nRatings)
    Syy = sum([i**2 for i in tempGroupList]) - pow(sum(tempGroupList),2)/float(nRatings)
    Sxy = sum(i*j for i, j in zip(tempRatingList, tempGroupList)) - sum(tempRatingList)*sum(tempGroupList)/float(nRatings)
    
    if Sxx != 0 and Syy != 0:
        pearsonCorrelationDict[name] = Sxy/sqrt(Sxx*Syy)
    else:
        pearsonCorrelationDict[name] = 0

In [ ]:
pearsonCorrelationDict

In [27]:
pearsonDF = pd.DataFrame.from_dict(pearsonCorrelationDict, orient='index')
pearsonDF.columns = ['similarityIndex']
pearsonDF['userId'] = pearsonDF.index
topUsers = pearsonDF.sort_values(by='similarityIndex', ascending=False)[0:50]

In [ ]:
topUsers

In [41]:
topUsersRating = topUsers.merge(ratings_df, on='userId', how='outer')  
print(topUsersRating.head())  

   similarityIndex userId  movieId  rating
0         0.155543   (6,)      NaN     NaN
1         1.000000  (18,)      NaN     NaN
2         0.000000  (42,)      NaN     NaN
3         0.000000  (44,)      NaN     NaN
4         0.000000  (45,)      NaN     NaN


In [42]:
print(topUsersRating.head())  

   similarityIndex userId  movieId  rating
0         0.155543   (6,)      NaN     NaN
1         1.000000  (18,)      NaN     NaN
2         0.000000  (42,)      NaN     NaN
3         0.000000  (44,)      NaN     NaN
4         0.000000  (45,)      NaN     NaN


In [43]:
topUsersRating

,similarityIndex,userId,movieId,rating
0,0.155543,"(6,)",NaN,NaN
1,1.000000,"(18,)",NaN,NaN
2,0.000000,"(42,)",NaN,NaN
3,0.000000,"(44,)",NaN,NaN
4,0.000000,"(45,)",NaN,NaN
...,...,...,...,...
100881,NaN,9,5962.0,1.0
100882,NaN,9,5965.0,4.0
100883,NaN,9,5988.0,2.0
100884,NaN,9,6001.0,4.0


In [46]:
topUsersRating = topUsers.merge(ratings_df, left_on='userId', right_on='userId', how='outer')
topUsersRating['weightedRating'] = topUsersRating['similarityIndex']*topUsersRating['rating']

In [47]:
topUsersRating

,similarityIndex,userId,movieId,rating,weightedRating
0,0.155543,"(6,)",NaN,NaN,NaN
1,1.000000,"(18,)",NaN,NaN,NaN
2,0.000000,"(42,)",NaN,NaN,NaN
3,0.000000,"(44,)",NaN,NaN,NaN
4,0.000000,"(45,)",NaN,NaN,NaN
...,...,...,...,...,...
100881,NaN,9,5962.0,1.0,NaN
100882,NaN,9,5965.0,4.0,NaN
100883,NaN,9,5988.0,2.0,NaN
100884,NaN,9,6001.0,4.0,NaN


In [48]:
tempTopUsersRating = topUsersRating.groupby('movieId').sum()[['similarityIndex','weightedRating']]
recommendation_df = pd.DataFrame()
recommendation_df['weighted average recommendation score'] = tempTopUsersRating['weightedRating']/tempTopUsersRating['similarityIndex']
recommendation_df['movieId'] = tempTopUsersRating.index

In [49]:
recommendation_df = recommendation_df.sort_values(by='weighted average recommendation score', ascending=False)
movies_df.loc[movies_df['movieId'].isin(recommendation_df.head(10)['movieId'].tolist())]

,movieId,title,genres,year
0,1,Toy Story,Animation,1995
1,2,Jumanji,Adventure,1995
2,3,Pulp Fiction,Crime,1994
3,4,Akira,Animation,1988
